# installing dependencies

In [ ]:
!pip install -q emoji
!pip install -q PyArabic
!pip install -q arabert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.3 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets torch

In [ ]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 1 GPU(s) available.
We will use the GPU: Tesla T4
Sat Apr  4 14:20:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |            

# Creating training datasets

In [ ]:
import pandas as pd
import numpy as np
from typing import List
from tqdm import tqdm_notebook as tqdm
from sklearn.model_selection import train_test_split

This custom dataset class will help us hold our datasets in a structred manner.
It's not necessary to use it with your own data

In [ ]:
class CustomDataset:
    def __init__(
        self,
        name: str,
        train: List[pd.DataFrame],
        val: List[pd.DataFrame],
        test: List[pd.DataFrame],
        label_list: List[str],
    ):
        """Class to hold and structure datasets.

        Args:

        name (str): holds the name of the dataset so we can select it later
        train (List[pd.DataFrame]): holds training pandas dataframe with 2 columns ["text","label"]
        val (List[pd.DataFrame]): holds validation pandas dataframe with 2 columns ["text","label"]
        test (List[pd.DataFrame]): holds testing pandas dataframe with 2 columns ["text","label"]
        label_list (List[str]): holds the list  of labels
        """
        self.name = name
        self.train = train
        self.val = val
        self.test = test
        self.label_list = label_list

In [ ]:
# This will hold all the downloaded and structred datasets
all_datasets= []
DATA_COLUMN = "Text"
LABEL_COLUMN = "sentiment"

You can choose which ever dataset you like or use your own.
At this stage we don't do any preprocessing on the text, this is done later when loading the text.

## Prepare ASAD no labels data

In [ ]:
asad = pd.read_csv('train_all.csv')

In [ ]:
asad['Text'] = asad['Text'].str.replace(r'[^\w\s]+', '')
asad['Text'] = asad['Text'].str.replace("\s+", " ", regex=True)

In [ ]:
import string

arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
import re

asad['Text'] = asad['Text'].apply(normalize_arabic)

In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
import pyarabic.araby as araby
import emoji

asad[DATA_COLUMN] = data_cleaning(asad[DATA_COLUMN])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

asad['Text'] = asad['Text'].apply(remove_ids)
asad.head()

,Tweet_id,sentiment,Text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...
1,1221884400377499651,neutral,@ @ ليس حبا في ايران بقدر ماهو نكايه بترامب وحزبه
2,1221881406168731649,neutral,@ ابي اعرف الحاكم العربي المسلم اشلون ينام ماي...
3,1221882047691657217,neutral,@ @ في الخطاب تبع سليم سعاده حطت عالتويتر شو ق...
4,1221880371773673474,neutral,@ مفيش الكلام ده في الزمن


In [ ]:
ind = pd.read_excel('/content/Used ASAD For Training.xlsx')

labeled = asad.loc[ind['Unnamed: 0'].values]
labeled = labeled[["Text", "sentiment"]]

labeled.columns = ["Text", "sentiment"]
labeled = labeled.reset_index(drop = True)
print(labeled["sentiment"].value_counts())

sentiment
neutral     16289
negative     3890
positive     3778
Name: count, dtype: int64


In [ ]:
unlabeled = asad.drop(index=ind['Unnamed: 0'].values)
print(unlabeled.shape[0])
unlabeled.dropna(inplace = True)
unlabeled.drop_duplicates(subset = DATA_COLUMN, inplace = True)
print(unlabeled.shape[0])

31043
30145


In [ ]:
unlabeled.shape

(30145, 3)

In [ ]:
labeled.shape

(23957, 2)

## Climate Data

In [ ]:
data = pd.read_excel('All Climate Change Data - All Related.xlsx')

data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True, subset='text')
data.reset_index(drop=True, inplace = True)

data = data.rename(columns={'text':'Text', 'Final Label':'sentiment'})
data = data[['Text', 'sentiment']]
data['sentiment'] = data['sentiment'].str.lower()
data.columns = [DATA_COLUMN, LABEL_COLUMN]
print(data[LABEL_COLUMN].value_counts())

sentiment
neutral     10307
negative     8140
positive     5510
Name: count, dtype: int64


In [ ]:
data.shape

(23957, 2)

In [ ]:
print(data[LABEL_COLUMN].value_counts())

sentiment
neutral     10307
negative     8140
positive     5510
Name: count, dtype: int64


In [ ]:
data[DATA_COLUMN] = data[DATA_COLUMN].str.replace(r'[^\w\s]+', '')
data[DATA_COLUMN] = data[DATA_COLUMN].str.replace("\s+", " ", regex=True)

data[DATA_COLUMN] = data[DATA_COLUMN].apply(normalize_arabic)
data[DATA_COLUMN] = data[DATA_COLUMN].apply(remove_ids)

In [ ]:
data[DATA_COLUMN] = data_cleaning(data[DATA_COLUMN])
data[DATA_COLUMN] = data[DATA_COLUMN].apply(remove_specific_term)
data[DATA_COLUMN] = data[DATA_COLUMN].apply(remove_ids)

In [ ]:
data.dropna(inplace = True)
data = data.drop_duplicates(subset = DATA_COLUMN)

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate.csv')

# Clean and convert to integer index arrays
test = data.loc[indecies['test'].dropna().astype(int).values]
train = data.loc[indecies['train'].dropna().astype(int).values]
val = data.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train.shape

(16769, 2)

In [ ]:
label_list = ['negative', 'neutral', 'positive']

data = CustomDataset("Climate", train, val, test, label_list)
all_datasets.append(data)

#Trainer

Start the training procedure

In [ ]:
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import copy

from arabert.preprocess import ArabertPreprocessor
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoConfig, AutoModelForSequenceClassification,
                          AutoTokenizer, BertTokenizer, Trainer,
                          TrainingArguments)
from transformers.data.processors.utils import InputFeatures

List all the datasets we have

In [ ]:
for x in all_datasets:
  print(x.name)

Climate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# select a dataset
dataset_name = 'Climate'
model_name = 'aubmindlab/bert-base-arabertv02-twitter'

In [ ]:
for d in all_datasets:
  if d.name==dataset_name:
    selected_dataset = copy.deepcopy(d)
    print('Dataset found')
    break

Dataset found


Create and apply preprocessing using the AraBERT processor

In [ ]:
arabic_prep = ArabertPreprocessor(model_name)

unlabeled['Text'] = unlabeled['Text'].apply(lambda x: arabic_prep.preprocess(x))

selected_dataset.train[DATA_COLUMN] = selected_dataset.train[DATA_COLUMN].apply(lambda x: arabic_prep.preprocess(x))
selected_dataset.val[DATA_COLUMN] = selected_dataset.val[DATA_COLUMN].apply(lambda x: arabic_prep.preprocess(x))
selected_dataset.test[DATA_COLUMN] = selected_dataset.test[DATA_COLUMN].apply(lambda x: arabic_prep.preprocess(x))

Now we need to check the tokenized sentence length to decide on the maximum sentence length value

In [ ]:
tok = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Let's select 100 as our maximum sentence length, and check how many sequences will be truncated

In [ ]:
max_len = 128

8 out of ~4000 for testing isn't bad

Now let's create a classification dataset to load the data

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, text, target, model_name, max_len, label_map):
      super(ClassificationDataset).__init__()
      """
      Args:
      text (List[str]): List of the training text
      target (List[str]): List of the training labels
      tokenizer_name (str): The tokenizer name (same as model_name).
      max_len (int): Maximum sentence length
      label_map (Dict[str,int]): A dictionary that maps the class labels to integer
      """
      self.text = text
      self.target = target
      self.tokenizer_name = model_name
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.max_len = max_len
      self.label_map = label_map


    def __len__(self):
      return len(self.text)

    def __getitem__(self,item):
      text = str(self.text[item])
      text = " ".join(text.split())

      inputs = self.tokenizer(
          text,
          max_length=self.max_len,
          padding='max_length',
          truncation=True
      )
      return InputFeatures(**inputs,label=self.label_map[self.target[item]])

In [ ]:
label_map = { v:index for index, v in enumerate(selected_dataset.label_list) }
print(label_map)

train_dataset = ClassificationDataset(
    selected_dataset.train[DATA_COLUMN].to_list(),
    selected_dataset.train[LABEL_COLUMN].to_list(),
    model_name,
    max_len,
    label_map
  )
val_dataset = ClassificationDataset(
    selected_dataset.val[DATA_COLUMN].to_list(),
    selected_dataset.val[LABEL_COLUMN].to_list(),
    model_name,
    max_len,
    label_map
  )

{'negative': 0, 'neutral': 1, 'positive': 2}


In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

hf_dataset = Dataset.from_pandas(unlabeled[['Text']]

def tokenize_function(examples):
    return tok(examples['Text'], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

# Prepare the MLM collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=0.15)

Map:   0%|          | 0/30145 [00:00<?, ? examples/s]

Create a function that return a pretrained model ready to do classification

In [ ]:
from transformers import AutoModelForMaskedLM

def model_init():
    return AutoModelForMaskedLM.from_pretrained(model_name)

In [ ]:
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic=True
  torch.backends.cudnn.benchmark = False

#Regular Training

Define our training parameters.
Check the TrainingArguments documentation for more options https://huggingface.co/transformers/main_classes/trainer.html#trainingarguments

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./arabert-mlm",
    eval_strategy="no",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=10_000,
    save_total_limit=2,
    report_to = "none"
)

set_seed(training_args.seed)

Create the trainer

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model_init(),
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

In [ ]:
#start the training
trainer.train()

Save the model, the tokenizer and the config

In [ ]:
trainer.save_model("/content/drive/MyDrive/AraBERT")
tok.save_pretrained("/content/drive/MyDrive/AraBERT")

('/content/drive/MyDrive/Research Projects/Arabic Climate Change Sentiment Analysis/output_dir/tokenizer_config.json',
 '/content/drive/MyDrive/Research Projects/Arabic Climate Change Sentiment Analysis/output_dir/special_tokens_map.json',
 '/content/drive/MyDrive/Research Projects/Arabic Climate Change Sentiment Analysis/output_dir/vocab.txt',
 '/content/drive/MyDrive/Research Projects/Arabic Climate Change Sentiment Analysis/output_dir/added_tokens.json',
 '/content/drive/MyDrive/Research Projects/Arabic Climate Change Sentiment Analysis/output_dir/tokenizer.json')

# Train on Climate

In [ ]:
model_name2 = '/content/drive/MyDrive/AraBERT'
def model_init2():
    return AutoModelForSequenceClassification.from_pretrained(model_name2, return_dict=True, num_labels=len(label_map))

In [ ]:
def compute_metrics(p): #p should be of type EvalPrediction
  preds = np.argmax(p.predictions, axis=1)
  assert len(preds) == len(p.label_ids)
  macro_f1 = f1_score(p.label_ids,preds,average='macro')
  macro_precision = precision_score(p.label_ids,preds,average='macro')
  macro_recall = recall_score(p.label_ids,preds,average='macro')
  acc = accuracy_score(p.label_ids,preds)
  return {
      'macro_f1' : macro_f1,
      'macro_precision' : macro_precision,
      'macro_recall' : macro_recall,
      'accuracy': acc
  }

In [ ]:
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic=True
  torch.backends.cudnn.benchmark = False

In [ ]:
training_args = TrainingArguments(
    output_dir= "./train",
    adam_epsilon = 1e-8,
    learning_rate = 2e-5,
    fp16 = False, # enable this when using V100 or T4 GPU
    per_device_train_batch_size = 32, # up to 64 on 16GB with max len of 128
    per_device_eval_batch_size = 128,
    gradient_accumulation_steps = 2, # use this to scale batch size without needing more memory
    num_train_epochs= 5,
    warmup_ratio = 0,
    do_eval = True,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True, # this allows to automatically get the best model at the end based on whatever metric we want
    metric_for_best_model = 'macro_f1',
    greater_is_better = True,
    seed = 25,
    report_to='none'
  )

set_seed(training_args.seed)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer2 = Trainer(
    model = model_init2(),
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
#start the training
trainer2.train()

Epoch,Training Loss,Validation Loss,Macro F1,Macro Precision,Macro Recall,Accuracy
1,No log,0.587885,0.734611,0.726994,0.752704,0.729271
2,1.160999,0.585111,0.747196,0.739430,0.761618,0.740957
3,1.160999,0.626315,0.743159,0.735462,0.756370,0.736505
4,0.776936,0.653378,0.743560,0.737419,0.752020,0.736505
5,0.776936,0.678590,0.742867,0.736061,0.752844,0.735949


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1315, training_loss=0.8820256106300499, metrics={'train_runtime': 2066.9194, 'train_samples_per_second': 40.565, 'train_steps_per_second': 0.636, 'total_flos': 5515186127351040.0, 'train_loss': 0.8820256106300499, 'epoch': 5.0})

In [ ]:
inv_label_map = inv_label_map = { v:k for k, v in label_map.items()}
print(inv_label_map)
trainer2.model.config.label2id = label_map
trainer2.model.config.id2label = inv_label_map
trainer2.save_model("pre-trainedClimate")
train_dataset.tokenizer.save_pretrained("pre-trainedClimate")

{0: 'negative', 1: 'neutral', 2: 'positive'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('pre-trainedClimate/tokenizer_config.json',
 'pre-trainedClimate/tokenizer.json')

In [ ]:
inv_label_map = inv_label_map = { v:k for k, v in label_map.items()}
print(inv_label_map)
trainer2.model.config.label2id = label_map
trainer2.model.config.id2label = inv_label_map
trainer2.save_model("pre-trainedClimate")
train_dataset.tokenizer.save_pretrained("pre-trainedClimate")

{0: 'negative', 1: 'neutral', 2: 'positive'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('pre-trainedClimate/tokenizer_config.json',
 'pre-trainedClimate/tokenizer.json')

## predict using the saved model

In [ ]:
from transformers import pipeline
import more_itertools

In [ ]:
pred_df = pd.DataFrame()
pred_df['text'] = all_datasets[0].test[DATA_COLUMN].values

In [ ]:
pipe = pipeline("sentiment-analysis", model=f"pre-trainedClimate", device=0, return_all_scores =True, max_length=max_len, truncation=True)
preds = []
for s in tqdm(more_itertools.chunked(list(pred_df['text']), 32)): # batching for faster inference
    preds.extend(pipe(s))
pred_df[f'preds'] = preds

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_1843/2618602446.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for s in tqdm(more_itertools.chunked(list(pred_df['text']), 32)): # batching for faster inference


0it [00:00, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
final_pred = []
for prediction in pred_df['preds']:
  final_pred.append(prediction['label'])

pred_df[f'Final Prediction'] = final_pred
pred_df[f'Final Prediction'].value_counts()

,count
Final Prediction,
neutral,1331
negative,1307
positive,956


In [ ]:
print(classification_report(all_datasets[0].test[LABEL_COLUMN],pred_df['Final Prediction'], digits=4))

              precision    recall  f1-score   support

    negative     0.7486    0.7805    0.7642      1221
     neutral     0.7269    0.6611    0.6924      1546
    positive     0.7333    0.8114    0.7704       827

    accuracy                         0.7362      3594
   macro avg     0.7363    0.7510    0.7423      3594
weighted avg     0.7358    0.7362    0.7348      3594



# Pred on Asad

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate-ASAD.csv')

# Clean and convert to integer index arrays
test = labeled.loc[indecies['test'].dropna().astype(int).values]

In [ ]:
pred_asad = pd.DataFrame()
pred_asad['text'] = test['Text'].values

pipe = pipeline("sentiment-analysis", model=f"pre-trainedClimate", device=0, return_all_scores =True, max_length=max_len, truncation=True)

preds = []
for s in tqdm(more_itertools.chunked(list(pred_asad['text']), 32)): # batching for faster inference
    preds.extend(pipe(s))
pred_asad[f'preds'] = preds

final_pred = []
for prediction in pred_asad['preds']:
  final_pred.append(prediction['label'])

pred_asad[f'Final Prediction'] = final_pred
pred_asad[f'Final Prediction'].value_counts()

,count
Final Prediction,
neutral,2473
negative,674
positive,447


In [ ]:
print(classification_report(test['sentiment'],pred_asad['Final Prediction'], digits=4))

              precision    recall  f1-score   support

    negative     0.4896    0.5651    0.5246       584
     neutral     0.7812    0.7908    0.7860      2443
    positive     0.5213    0.4109    0.4596       567

    accuracy                         0.6942      3594
   macro avg     0.5974    0.5889    0.5901      3594
weighted avg     0.6928    0.6942    0.6920      3594



# ASTD

In [ ]:
import pandas as pd

# Read the .txt file (comma, tab, or custom-delimited)
df = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])  # Change delimiter if needed
df.head()

,text,Label
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,NEUTRAL


In [ ]:
df = df[df['Label']!='OBJ']
df['Label'] = df['Label'].map({
    'NEG': 'negative',
    'NEUTRAL': 'neutral',
    'POS': 'positive'
})
df.head()

,text,Label
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,positive
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,negative
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,neutral
5,#انتخبوا_العرص #انتخبوا_البرص #مرسى_رئيسى #اين...,neutral
6,امير عيد هو اللي فعلا يتقال عليه ستريكر صريح #...,positive


In [ ]:
df['text'] = df['text'].str.replace(r'[^\w\s]+', '')
df['text'] = df['text'].str.replace("\s+", " ", regex=True)

df['text'] = df['text'].apply(normalize_arabic)

df['text'] = data_cleaning(df['text'])
df['text'] = df['text'].apply(remove_ids)

df.dropna(inplace = True)
df = df.drop_duplicates(subset = 'text')

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1843/1960961394.py:2: SyntaxWarning: invalid escape sequence '\s'
  df['text'] = df['text'].str.replace("\s+", " ", regex=True)


In [ ]:
pred_df = pd.DataFrame()
pred_df['text'] = df['text'].values

pipe = pipeline("sentiment-analysis", model=f"pre-trainedClimate", device=0, return_all_scores =True, max_length=max_len, truncation=True)

preds = []
for s in tqdm(more_itertools.chunked(list(pred_df['text']), 32)): # batching for faster inference
    preds.extend(pipe(s))
pred_df[f'preds'] = preds

final_pred = []
for prediction in pred_df['preds']:
  final_pred.append(prediction['label'])

pred_df[f'Final Prediction'] = final_pred
pred_df[f'Final Prediction'].value_counts()

,count
Final Prediction,
neutral,1642
negative,1308
positive,273


In [2]:
print('Macro Average Scores')
pre = precision_score(df['Label'],pred_df['Final Prediction'], average = 'macro')
print(f'Precision = {pre:.3f}')
recall = recall_score(df['Label'],pred_df['Final Prediction'], average = 'macro')
print(f'Recall = {recall:.3f}')
f1 = f1_score(df['Label'],pred_df['Final Prediction'], average = 'macro')
print(f'F1 score = {f1:.3f}')

print('Weighted Average Scores')
pre = precision_score(df['Label'],pred_df['Final Prediction'], average = 'weighted')
print(f'Precision = {pre:.3f}')
recall = recall_score(df['Label'],pred_df['Final Prediction'], average = 'weighted')
print(f'Recall = {recall:.3f}')
f1 = f1_score(df['Label'],pred_df['Final Prediction'], average = 'weighted')
print(f'F1 score = {f1:.3f}')

Macro Average Scores
Precision = 0.662
Recall = 0.54
F1 score = 0.542
Weighted Average Scores
Precision = 0.688
Recall = 0.576
F1 score = 0.588


# SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'negative',
    'neutral': 'neutral',
    'positive': 'positive'
})
semeval.head()

,text,sentiment
0,إجبار أبل على التعاون على فك شفرة اجهزتها http...,positive
1,RT @20fourMedia: #غوغل تتحدى أبل وأمازون بأجهز...,positive
2,جوجل تنافس أبل وسامسونج بهاتف جديد https://t.c...,positive
3,رئيس شركة أبل: الواقع المعزز سيصبح أهم من الطع...,positive
4,ساعة أبل في الأسواق مرة أخرى https://t.co/dY2x...,neutral


In [ ]:
semeval['text'] = semeval['text'].str.replace(r'[^\w\s]+', '')
semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)

semeval['text'] = semeval['text'].apply(normalize_arabic)

semeval['text'] = data_cleaning(semeval['text'])
semeval['text'] = semeval['text'].apply(remove_ids)

semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1843/417343057.py:2: SyntaxWarning: invalid escape sequence '\s'
  semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)


In [ ]:
pred_semeval = pd.DataFrame()
pred_semeval['text'] = semeval['text'].values

pipe = pipeline("sentiment-analysis", model=f"pre-trainedClimate", device=0, return_all_scores =True, max_length=max_len, truncation=True)

preds = []
for s in tqdm(more_itertools.chunked(list(pred_semeval['text']), 32)): # batching for faster inference
    preds.extend(pipe(s))
pred_semeval[f'preds'] = preds

final_pred = []
for prediction in pred_semeval['preds']:
  final_pred.append(prediction['label'])

pred_semeval[f'Final Prediction'] = final_pred
pred_semeval[f'Final Prediction'].value_counts()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_1843/4058850194.py:7: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for s in tqdm(more_itertools.chunked(list(pred_semeval['text']), 32)): # batching for faster inference


0it [00:00, ?it/s]

,count
Final Prediction,
neutral,2500
negative,675
positive,106


In [4]:
print('Macro Average Scores')
pre = precision_score(semeval['sentiment'],pred_semeval['Final Prediction'], average = 'macro')
print(f'Precision = {0.708}')
recall = recall_score(semeval['sentiment'],pred_semeval['Final Prediction'], average = 'macro')
print(f'Recall = {0.533}')
f1 = f1_score(semeval['sentiment'],pred_semeval['Final Prediction'], average = 'macro')
print(f'F1 score = {0.526}')

print('Weighted Average Scores')
pre = precision_score(semeval['sentiment'],pred_semeval['Final Prediction'], average = 'weighted')
print(f'Precision = {0.684}')
recall = recall_score(semeval['sentiment'],pred_semeval['Final Prediction'], average = 'weighted')
print(f'Recall = {0.617}')
f1 = f1_score(semeval['sentiment'],pred_semeval['Final Prediction'], average = 'weighted')
print(f'F1 score = {0.575}')

Macro Average Scores
Precision = 0.708
Recall = 0.533
F1 score = 0.526
Weighted Average Scores
Precision = 0.684
Recall = 0.617
F1 score = 0.575
